## CMAPSS Predictive Maintenance - ML Pipeline
### Notebook 1: Preprocessing

In [0]:
from pyspark.sql.functions import(
  col, when, lit, min as spark_min, max as spark_max
)
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

In [0]:
"""RUL cap based on CMAPSS standard practice
Engines with RUL > 125 are all "healthy" - no urgency distinction needed """

RUL_CAP = 125

# Key sensors used for rolling features in Silver
KEY_SENSORS = ["sensor_2", "sensor_3", "sensor_4", "sensor_7",
               "sensor_11", "sensor_12", "sensor_15", "sensor_17",
               "sensor_20", "sensor_21"]

# All 21 sensors
ALL_SENSORS = [f"sensor_{i+1}" for i in range(21)]


In [0]:
silver_train = spark.read.format("delta").table("cmapss_project.silver.silver_train")
silver_test = spark.read.format("delta").table("cmapss_project.silver.silver_test")

print(f"silver_train: {silver_train.count()} rows x {len(silver_train.columns)} columns")
print(f"silver_test: {silver_test.count()} rows x {len(silver_test.columns)} columns")

silver_train: 160359 rows x 69 columns
silver_test: 104897 rows x 69 columns


In [0]:
"""Fill Null Rolling Std Values
    - fill nulls in rolling std columns with 0
    - nulls are caused by insufficient data points for rolling window
    - 0 = no volatility observed yet; semantically correct for cycle 1
"""
std_cols = [f"{s}_roll_std_10" for s in KEY_SENSORS]

for col_name in std_cols:
    silver_train = silver_train.withColumn(
        col_name, when(col(col_name).isNull(), 0.0).otherwise(col(col_name))
    )
    silver_test = silver_test.withColumn(
        col_name, when(col(col_name).isNull(), 0.0).otherwise(col(col_name))
    )

In [0]:
"""Cap RUL at 125 cycles
    - Engines with RUL > 125 are labeled as 125
    - Focuses model on learning the critical degradation window
    - Standard practice in CMAPSS literature
"""

silver_train = silver_train.withColumn(
    "rul_capped", when(col("rul") > RUL_CAP, RUL_CAP).otherwise(col("rul"))
)

# Verify the cap worked
print("RUL distribution before and after capping (train):")
silver_train.select(
    F.round(F.avg("rul"), 1).alias("avg_rul_original"),
    F.round(F.avg("rul_capped"), 1).alias("avg_rul_capped"),
    spark_max("rul").alias("max_rul_original"),
    spark_max("rul_capped").alias("max_rul_capped")
).show()

RUL distribution before and after capping (train):
+----------------+--------------+----------------+--------------+
|avg_rul_original|avg_rul_capped|max_rul_original|max_rul_capped|
+----------------+--------------+----------------+--------------+
|           122.3|          90.2|             542|           125|
+----------------+--------------+----------------+--------------+



In [0]:
"""Add RUL Zone Labels (for Classification)
    - Similar to Gold layer bucketing
    - THis is the classification target variable
    - Uses capped RUL so zones align with model focus window
"""

def assign_zone(rul_col):
    return(
        when(rul_col <=25, "1_Critical")
        .when(rul_col <= 50, "2_Warning")
        .when(rul_col <= 100, "3_Moderate")
        .when(rul_col <= 125, "4_Healthy")
        .otherwise("5_Healthy")
    )

silver_train = silver_train.withColumn("rul_zone", assign_zone(col("rul_capped")))

In [0]:
# Check zone distribution
print("Zone distribution in training data:")
silver_train.groupBy("rul_zone").count().orderBy("rul_zone").show()

Zone distribution in training data:
+----------+-----+
|  rul_zone|count|
+----------+-----+
|1_Critical|18434|
| 2_Warning|17725|
|3_Moderate|35450|
| 4_Healthy|88750|
+----------+-----+



In [0]:
"""Encode dataset_id as integer feature
    - FD001 : 1
    - FD002 : 2
    - FD003 : 3
    - FD004 : 4
"""

dataset_if_map = {"FD001": 1, "FD002": 2, "FD003": 3, "FD004": 4}

for df_name, df in [("train", silver_train), ("test", silver_test)]:
    df = df.withColumn(
        "dataset_id_encoded",
        when(col("dataset_id") == "FD001", 1)
        .when(col("dataset_id") == "FD002", 2)
        .when(col("dataset_id") == "FD003", 3)
        .otherwise(4)
    )
    
    if df_name == "train":
        silver_train = df
    else:
        silver_test = df


In [0]:
"""Define final feature set for ML"""

# Rolling feature column names
roll_mean_5_cols = [f"{s}_roll_mean_5" for s in KEY_SENSORS]
roll_mean_10_cols = [f"{s}_roll_mean_10" for s in KEY_SENSORS]
roll_mean_30_cols = [f"{s}_roll_mean_30" for s in KEY_SENSORS]
roll_std_10_cols = [f"{s}_roll_std_10" for s in KEY_SENSORS]

FEATURE_COLS = (
    ["dataset_id_encoded", "op_setting_1", "op_setting_2", "op_setting_3"] +
    ALL_SENSORS +
    roll_mean_5_cols + roll_mean_10_cols + roll_mean_30_cols +
    roll_std_10_cols
)

# Target Columns
REGRESSION_TARGET = "rul_capped"
CLASSIFICATION_TARGET = "rul_zone"

print(f"Total features: {len(FEATURE_COLS)}")



Total features: 65


In [0]:
"""Build Final ML Train & Test Tables
    - Train : all rows, all features + both targets
    - Test : only last cycle per engine (that's where RUL is labeled) 
"""
# Train
ml_train = silver_train.select(
    ["unit_id", "dataset_id", "cycle"] +
    FEATURE_COLS +
    [REGRESSION_TARGET, CLASSIFICATION_TARGET]
)

# Test
from pyspark.sql import Window
window_last = Window.partitionBy("unit_id", "dataset_id")

ml_test = (
    silver_test
    .withColumn("max_cycle", spark_max("cycle").over(window_last))
    .filter(col("cycle") == col("max_cycle"))
    .drop("max_cycle")
)

# Cap test RUL as well
ml_test = ml_test.withColumn(
    "rul_capped",
    when(col("rul") > RUL_CAP, RUL_CAP). otherwise(col("rul"))
)

ml_test = ml_test.withColumn("rul_zone", assign_zone(col("rul_capped")))

ml_test = ml_test.select(
    ["unit_id", "dataset_id", "cycle"] +
    FEATURE_COLS +
    [REGRESSION_TARGET, CLASSIFICATION_TARGET]
)

print(f"ml_train rows: {ml_train.count()}")
print(f"ml_test rows:  {ml_test.count()} (one row per engine)")

ml_train rows: 160359
ml_test rows:  707 (one row per engine)


In [0]:
(
    ml_train.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("cmapss_project.gold.gold_ml_train")

)

(
    ml_test.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("cmapss_project.gold.gold_ml_test")
)

print(f"gold_ml_train: {ml_train.count()} rows x {len(ml_train.columns)} columns")
print(f"gold_ml_test:  {ml_test.count()} rows x {len(ml_test.columns)} columns")

gold_ml_train: 160359 rows x 70 columns
gold_ml_test:  707 rows x 70 columns
